In [2]:
from pathlib import Path
BASE = Path("/home/lka/HYBRID_SARIMA")
print(BASE.exists(), BASE)

#Import the file to read it
import pandas as pd
disocupati_trimestral = pd.read_csv(BASE / 'data.csv', 
                       index_col='Period', 
                       parse_dates=True)
print(disocupati_trimestral)

True /home/lka/HYBRID_SARIMA
             15-34 M    15-34 F   35-49 M   35-49 F   50-74 M   50-74 F  \
Period                                                                    
2004-Q1    11.720628  17.198032  4.283283  8.776062  4.004764  5.376203   
2004-Q2    11.410664  16.187372  3.654493  7.196378  3.636536  4.515018   
2004-Q3    10.041510  15.682014  3.520125  6.945425  3.394804  3.976243   
2004-Q4    11.918557  17.227696  4.020448  7.582207  3.604743  3.583677   
2005-Q1    12.334781  16.852636  4.316241  7.577546  3.440298  4.204936   
...              ...        ...       ...       ...       ...       ...   
2025-Q2    11.341464  13.635416  5.106017  7.025122  3.776334  3.889283   
2025-Q3     9.628072  11.355385  4.656108  6.239399  3.108541  3.482718   
2025-Q4    10.903735  10.928362  3.989899  5.704370  3.250742  3.129547   
2026-Q1     9.534584  10.900056  4.316380  5.611548  2.984988  3.392216   
2026-Q2    10.244865  11.855438  5.044973  6.388627  3.268385  3.460483

/tmp/ipykernel_207877/3099411105.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  disocupati_trimestral = pd.read_csv(BASE / 'data.csv',


In [3]:
#Dividir el dataset en training, validation y test
series = [
    '15-34 M', '15-34 F',
    '35-49 M', '35-49 F',
    '50-74 M', '50-74 F',
    'Global_Rate'
]

# Ordenar cronológicamente
df = disocupati_trimestral

# División 70% Train, 15% Validation, 15% Test
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df[series].iloc[:train_end].copy()
val = df[series].iloc[train_end:val_end].copy()
test = df[series].iloc[val_end:].copy()

# Mostrar resultados para cada serie
#for serie in series:
#    print(f"\nSerie: {serie}")
#    print(f"Train: {len(train[serie])} observaciones")
#    print(f"Validation: {len(val[serie])} observaciones")
#    print(f"Test: {len(test[serie])} observaciones")

print(train)
print(val)
print(test)


             15-34 M    15-34 F   35-49 M    35-49 F   50-74 M   50-74 F  \
Period                                                                     
2004-Q1    11.720628  17.198032  4.283283   8.776062  4.004764  5.376203   
2004-Q2    11.410664  16.187372  3.654493   7.196378  3.636536  4.515018   
2004-Q3    10.041510  15.682014  3.520125   6.945425  3.394804  3.976243   
2004-Q4    11.918557  17.227696  4.020448   7.582207  3.604743  3.583677   
2005-Q1    12.334781  16.852636  4.316241   7.577546  3.440298  4.204936   
...              ...        ...       ...        ...       ...       ...   
2018-Q2    17.908050  21.321705  7.759217  10.435680  6.683032  6.283515   
2018-Q3    16.802144  19.343542  6.191831   9.120602  5.538136  5.299045   
2018-Q4    19.322895  22.132355  7.396393  10.673055  6.327076  6.189145   
2019-Q1    19.163606  21.713279  8.111591  11.250865  6.697788  6.034775   
2019-Q2    17.128382  19.347371  7.161324   9.818228  5.738406  5.356958   

           

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

pdq=list(product(range(3),repeat=3))
seasonal_pdq=[(P,D,Q,4) for P,D,Q in pdq]
sarima_grid=list(product(pdq,seasonal_pdq))

architectures=[(32,),(64,),(128,),(32,32),(64,32),(64,64),(128,32),(128,64),(128,128)]  #
lstm_grid=list(product(architectures,[0.01,0.001],[4,6,8],[100],[16,32],[0.1,0.3,0.5]))

class LSTM(nn.Module):
    def __init__(self,neurons,dropout):
        super().__init__()
        self.layers=nn.ModuleList()
        size=1
        for n in neurons:
            self.layers.append(nn.LSTM(size,n,batch_first=True))
            size=n
        self.drop=nn.Dropout(dropout)
        self.fc=nn.Linear(size,1)

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x, _ = layer(x)
            if i == 0:
                x = self.drop(x)
        return self.fc(x[:, -1, :])

def sequences(z,w,start,end):
    X=np.array([z[i-w:i] for i in range(start,end)],dtype=np.float32)
    y=np.array(z[start:end],dtype=np.float32)
    return torch.tensor(X.reshape(-1,w,1),device=device),torch.tensor(y.reshape(-1,1),device=device)

def train_lstm(res_train,res_val,sarima_val,y_val,config,final=False,best_epoch=None):
    neurons,lr,w,epochs,batch,dropout=config
    torch.manual_seed(42)
    scaler=MinMaxScaler()
    a=scaler.fit_transform(res_train.reshape(-1,1)).ravel()

    if final:
        z=a
    else:
        b=scaler.transform(res_val.reshape(-1,1)).ravel()
        z=np.concatenate([a,b])

    X,y=sequences(z,w,w,len(a) if not final else len(z))
    model=LSTM(neurons,dropout).to(device)
    optimizer=torch.optim.Adam(model.parameters(),lr=lr)
    criterion=nn.MSELoss()
    best=np.inf
    best_ep=0
    wait=0

    if not final:
        Xv,_=sequences(z,w,len(a),len(z))

    for epoch in range(best_epoch if final else epochs):
        model.train()
        idx=torch.randperm(len(X),device=device)

        for ids in idx.split(batch):
            optimizer.zero_grad()
            loss=criterion(model(X[ids]),y[ids])
            loss.backward()
            optimizer.step()

        if not final:
            model.eval()
            with torch.no_grad():
                pred=model(Xv).cpu().numpy()

            residual_pred=scaler.inverse_transform(pred).ravel()
            mse=mean_squared_error(y_val,sarima_val+residual_pred)

            if mse<best-1e-10:
                best=mse
                best_ep=epoch+1
                wait=0
            else:
                wait+=1

            if wait>=15:
                break

    if final:
        model.eval()
        return model,scaler

    return best,best_ep,epoch+1

grid_results=[]
results=[]
best_params={}
hybrid_predictions={}

for col in train.columns:
    y_train=train[col].astype(float).reset_index(drop=True)
    y_val=val[col].astype(float).reset_index(drop=True)
    y_test=test[col].astype(float)
    y_train_val=pd.concat([y_train,y_val],ignore_index=True)

    best_mse=np.inf
    best_sarima=None
    best_lstm=None
    best_epoch=0

    for order,seasonal in sarima_grid:
        try:
            sarima=SARIMAX(
                y_train,
                order=order,
                seasonal_order=seasonal,
                enforce_stationarity=True,
                enforce_invertibility=True
            ).fit(disp=False,maxiter=200)

            if not sarima.mle_retvals["converged"]:
                continue

            res_train=sarima.resid.to_numpy()[sarima.loglikelihood_burn:]
            sarima_val=np.asarray(sarima.forecast(steps=len(y_val)))
            res_val=y_val.to_numpy()-sarima_val

            if len(res_train)<10 or not np.all(np.isfinite(res_train)) or not np.all(np.isfinite(res_val)):
                continue

        except (ValueError,np.linalg.LinAlgError):
            continue

        for config in lstm_grid:
            mse,epoch,trained=train_lstm(res_train,res_val,sarima_val,y_val.to_numpy(),config)
            neurons,lr,w,epochs,batch,dropout=config

            grid_results.append([
                col,order,seasonal,len(neurons),str(neurons),
                lr,w,epochs,epoch,trained,batch,dropout,mse
            ])

            if mse<best_mse:
                best_mse=mse
                best_sarima=(order,seasonal)
                best_lstm=config
                best_epoch=epoch

    if best_sarima is None:
        raise ValueError(f"No valid model for {col}")

    best_params[col]={
        "SARIMA":best_sarima,
        "LSTM":best_lstm,
        "Best_Epoch":best_epoch,
        "MSE_Validation":best_mse
    }

    sarima=SARIMAX(
        y_train_val,
        order=best_sarima[0],
        seasonal_order=best_sarima[1],
        enforce_stationarity=True,
        enforce_invertibility=True
    ).fit(disp=False,maxiter=200)

    residuals=sarima.resid.to_numpy()[sarima.loglikelihood_burn:]
    lstm,scaler=train_lstm(residuals,None,None,None,best_lstm,final=True,best_epoch=best_epoch)

    sarima_pred=np.asarray(sarima.forecast(steps=len(y_test)))
    history=list(residuals)
    lstm_pred=[]

    for i in range(len(y_test)):
        w=best_lstm[2]
        x=scaler.transform(np.array(history[-w:]).reshape(-1,1))
        x=torch.tensor(x.reshape(1,w,1),dtype=torch.float32,device=device)

        with torch.no_grad():
            pred=lstm(x).cpu().numpy()

        pred=scaler.inverse_transform(pred)[0,0]
        lstm_pred.append(pred)
        history.append(float(y_test.iloc[i])-sarima_pred[i])

    lstm_pred=np.array(lstm_pred)
    hybrid_pred=sarima_pred+lstm_pred
    real=y_test.to_numpy()

    mae=mean_absolute_error(real,hybrid_pred)
    mse=mean_squared_error(real,hybrid_pred)
    rmse=np.sqrt(mse)
    mape=np.mean(np.abs((real-hybrid_pred)/real))*100 if np.all(real!=0) else np.nan
    values=y_train_val.to_numpy()
    scale=np.mean(np.abs(values[4:]-values[:-4]))
    mase=mae/scale if scale>0 else np.nan
    r2=r2_score(real,hybrid_pred)

    results.append([col,mae,mse,rmse,mape,mase,r2])

    hybrid_predictions[col]=pd.DataFrame({
        "Observed":real,
        "SARIMA":sarima_pred,
        "LSTM_Residual":lstm_pred,
        "Hybrid":hybrid_pred
    },index=y_test.index)

    print(col,"Validation MSE:",round(best_mse,4),"Test MSE:",round(mse,4))

grid_df=pd.DataFrame(grid_results,columns=[
    "Serie","Order","Seasonal_Order","Layers","Neurons",
    "Learning_Rate","Window","Epochs","Best_Epoch",
    "Epochs_Trained","Batch_Size","Dropout","MSE"
])

grid_df=grid_df.sort_values(["Serie","MSE"]).reset_index(drop=True)
grid_df["Rank"]=grid_df.groupby("Serie").cumcount()+1
top10=grid_df[grid_df["Rank"]<=10]

results_df=pd.DataFrame(results,columns=[
    "Serie","MAE","MSE","RMSE","MAPE","MASE","R2"
])

output=Path("results/SARIMA_LSTM_Joint")
output.mkdir(parents=True,exist_ok=True)
for col, pred in hybrid_predictions.items():
    pred.to_csv(output/f"pred_{col.replace(' ', '_')}.csv")

#grid_df.to_csv(output/"SARIMA_LSTM_Joint_Grid.csv",index=False)
top10.to_csv(output/"SARIMA_LSTM_Joint_Top10.csv",index=False)
results_df.to_csv(output/"SARIMA_LSTM_Joint_Test.csv",index=False)

print(results_df.round(4))
print(best_params)
